# RAG 기본 프로세스

### 라이브러리 선언

In [1]:
import os
from dotenv import load_dotenv

# 환경 변수 로드 (.env)
load_dotenv()

# RAG 프로세스 핵심 라이브러리
import langchain, chromadb, pypdf
# PDF 로더
from langchain_community.document_loaders import PyPDFLoader
# 텍스트 청크 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 임베딩모델 OpenAI 라이브러리
from langchain_openai import OpenAIEmbeddings
# 임베딩모델 무료 라이브러리 (허깅페이스)
from langchain_community.embeddings import HuggingFaceEmbeddings
# 벡터 DB 생성 라이브러리
from langchain_community.vectorstores import Chroma
# 모델 GPT 사용 라이브러리
from langchain_openai import ChatOpenAI
# 모델 GGUF 사용 라이브러리
from langchain_community.llms import LlamaCpp
# 모델 OLLAMA 사용 라이브러리
from langchain_community.llms import Ollama
# 임베딩모델 Ollama 라이브러리
from langchain_community.embeddings import OllamaEmbeddings
# 프롬프트 템플릿 라이브러리
from langchain_core.prompts import PromptTemplate
# RAG 체인 구성 라이브러리
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

### 환경 설정

In [2]:
# 폴더 구조 설정
DATA_DIR = "dataset"
PDF_PATH = os.path.join(DATA_DIR, "모집요강.pdf")
PERSIST_DIR = "vector_db"
MODEL_DIR = "models"
OPENAI_APIKEY = os.getenv("OPENAI_API_KEY")

# LLM 모델 선택: "gpt" 또는 "ollama"
MODEL_TYPE = "ollama"   # "gpt" 또는 "ollama" 중 선택
OLLAMA_MODEL = "gemma4:e2b"  # Ollama 사용 시 모델명

# 폴더 생성
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("데이터 폴더:", DATA_DIR)
print("PDF 경로:", PDF_PATH)
print("저장 경로:", PERSIST_DIR)

데이터 폴더: dataset
PDF 경로: dataset\모집요강.pdf
저장 경로: vector_db


In [3]:
# 설정값 정의
CHUNK_SIZE = 1000        # 청크 크기 (권장: 600-1200)
CHUNK_OVERLAP = 250     # 청크 오버랩 (권장: 100-200)
K_RETRIEVAL = 4         # 검색할 문서 수 (권장: 3-6)

print("설정된 하이퍼파라미터:")
print(f"청크 크기: {CHUNK_SIZE}")
print(f"오버랩: {CHUNK_OVERLAP}")
print(f"검색 수: {K_RETRIEVAL}")

설정된 하이퍼파라미터:
청크 크기: 1000
오버랩: 250
검색 수: 4


# 1. 데이터 준비

### 1-1. Data Load

In [4]:
# PDF 로딩 실행
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f" {len(docs)}개 페이지 로딩 완료!")

 20개 페이지 로딩 완료!


### 1-2. Split

In [5]:
# 텍스트 청크 분할 실행
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)

chunks = text_splitter.split_documents(docs)
print(f"생성된 청크 수: {len(chunks)}")

생성된 청크 수: 42


### 1-3. 임베딩 Store

In [6]:
USE_OPENAI_EMBEDDING = True

if MODEL_TYPE == "ollama":
    print("Ollama bge-m3 임베딩 모델 로딩...")
    embedding_model = OllamaEmbeddings(model="bge-m3")
elif USE_OPENAI_EMBEDDING:
    print("OpenAI 임베딩 모델 로딩...")
    os.environ["OPENAI_API_KEY"] = OPENAI_APIKEY
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    print("한국어 임베딩 무료 모델 로딩...")
    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

Ollama bge-m3 임베딩 모델 로딩...


C:\Users\junky\AppData\Local\Temp\ipykernel_30708\402772301.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding_model = OllamaEmbeddings(model="bge-m3")


In [7]:
# ChromaDB 생성 및 벡터 저장
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": K_RETRIEVAL}
)

# 2. 정보 검색 및 텍스트 생성

In [8]:
TEMPERATURE = 0.2

if MODEL_TYPE == "gpt":
    os.environ["OPENAI_API_KEY"] = OPENAI_APIKEY
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=TEMPERATURE)
elif MODEL_TYPE == "gguf":
    model_path = os.path.join(MODEL_DIR, "llama-model.gguf")
    llm = LlamaCpp(model_path=model_path, n_ctx=4096, n_threads=2, temperature=TEMPERATURE)
elif MODEL_TYPE == "ollama":
    llm = Ollama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

C:\Users\junky\AppData\Local\Temp\ipykernel_30708\340754850.py:10: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model=OLLAMA_MODEL, temperature=TEMPERATURE)


In [9]:
# RAG용 프롬프트 템플릿 정의
PROMPT_TEMPLATE = """
다음 컨텍스트만을 사용해서 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하고, 추측하지 마세요.
가능하면 출처 정보(페이지 번호)도 함께 제공하세요.

컨텍스트:
{context}

질문: {question}
답변:
"""

prompt_template = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

In [10]:
# RAG 체인 구성
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        page = doc.metadata.get('page', '알 수 없음')
        formatted.append(f"[출처 {i}] 페이지 {page}\n{content}")
    return "\n\n".join(formatted)

retrieval_chain = RunnableParallel(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "source_documents": retriever
    }
)

answer_chain = (
    {
        "context": lambda x: x["context"],
        "question": lambda x: x["question"]
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

rag_chain = retrieval_chain | RunnablePassthrough.assign(answer=answer_chain)

### 질의 테스트

In [11]:
question = "스마트금융과 모집 일정은?"
full_result = rag_chain.invoke(question)
print(f"\n질문: {question}")
print(f"\n답변:\n{full_result['answer']}")


질문: 스마트금융과 모집 일정은?

답변:
제공된 컨텍스트에는 스마트금융과 모집 일정에 대한 정보가 없습니다.


### FAST API 연동

In [12]:
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio

app = FastAPI(title="ML API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["*"],
)

class InDataset(BaseModel):
    question : str

@app.post("/predict", status_code=200)
async def predict_tf(x: InDataset):
    response = rag_chain.invoke(x.question)
    return {"prediction": response['answer'], "references": response['source_documents'] }

@app.get('/')
async def root():
    return {"message": "online"}

In [ ]:
from pyngrok import ngrok
import asyncio
import uvicorn
import nest_asyncio

# 1. Ngrok 설정
auth_token = os.getenv("NGROK_AUTH_TOKEN")
if auth_token:
    try:
        ngrok.set_auth_token(auth_token)
        tunnels = ngrok.get_tunnels()
        for t in tunnels:
            ngrok.disconnect(t.public_url)
        
        ngrokTunnel = ngrok.connect(8000)
        print(f"\n[Ngrok] 공용 URL: {ngrokTunnel.public_url}")
    except Exception as e:
        print(f"[Ngrok] 설정 건너뜀: {e}")

# 2. Jupyter 환경을 위한 중첩 루프 허용
nest_asyncio.apply()

# 3. 서버 설정 및 실행 (Jupyter용 비동기 방식)
print("\n[FastAPI] 서버를 시작합니다. (중단하려면 커널을 중지하세요)")
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Jupyter 셀에서는 await를 직접 사용할 수 있습니다.
await server.serve()

INFO:     Started server process [30708]



[Ngrok] 공용 URL: https://grandparental-nimbly-randee.ngrok-free.dev

[FastAPI] 서버를 시작합니다. (중단하려면 커널을 중지하세요)


INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     125.178.72.49:0 - "GET / HTTP/1.1" 200 OK
INFO:     125.178.72.49:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     125.178.72.49:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     125.178.72.49:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     125.178.72.49:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     125.178.72.49:0 - "POST /predict HTTP/1.1" 200 OK
